<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Where dense search fails

**[Where dense search fails](https://learning.nextia-ai.com/courses/search/m05/where-dense-search-fails/)** · Search and Ranking: Find the Best Match · Module 5, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure dense search against BM25 query by query, find the slices where it fails, and read the queries behind the averages.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 40 MB). No account and no key. |
| **You should know** | [Search by meaning](https://learning.nextia-ai.com/courses/search/m05/search-by-meaning/), and paired differences by slice. |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m05/where-dense-search-fails/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M05-L02-where-dense-search-fails/where-dense-search-fails-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 41 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/embeddings/products.npy": "f422870c4ab75364fad883fa8858bd7f31f2591bad07753a8f713a0872824ab0",
    "data/embeddings/queries.npy": "5333da6bd9ac44ec1bf92ddf916ec60cb557466b90b5075711f59c8e61f96253",
    "data/embeddings/model.json": "b5d1088cde17e0eb1bb9565f2e1e45c8cfb5b6a545ae72e44404d2429d6d9af4",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two searches, query by query

The next cell builds a plain BM25 over one field (the title, the French title and the description) and loads the shipped vectors for dense search. It is simpler than the project's best lexical run: no stemming, no spelling correction, no synonyms and no code rule. So your numbers will differ from the lesson's, but the pattern is what matters. Run it (about 20 seconds).

In [ ]:
import re, bm25s
cat = catalogue()
q = queries()
j = judgments()
ids = cat["product_id"].to_numpy()
P = np.load("data/embeddings/products.npy").astype(np.float32)   # one row per product
Q = np.load("data/embeddings/queries.npy").astype(np.float32)    # one row per query of queries.csv

def tokens(text):
    return [t for t in re.split(r"[^\w]+|_", str(text).lower()) if t]

# One BM25 field: the title, the French title (machine-translated, for some products) and the description.
corpus = [tokens(t) for t in cat["title"] + " " + cat["title_fr"] + " " + cat["description"]]
lex = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
lex.index(corpus, show_progress=False)

def top(score, k=10):
    """The k best product IDs (ties by product ID)."""
    idx = np.argpartition(-score, k)[:k]
    return list(ids[idx[np.lexsort((ids[idx], -score[idx]))]])

def bm25_top(text, k=10):
    words = [w for w in tokens(text) if w in lex.vocab_dict]
    return top(lex.get_scores(words), k) if words else []

def dense_top(row, k=10):
    return top(P @ Q[row], k)

print(f"{len(ids):,} products indexed; {P.shape[1]} numbers per vector")

Run the next cell. It searches every validation query both ways and keeps NDCG@10 per query, with the query's source and kind.

In [ ]:
val = q[q["split"] == "validation"]
run_bm25 = {qid: bm25_top(text) for qid, text in zip(val["query_id"], val["query"])}
run_dense = {qid: dense_top(row) for qid, row in zip(val["query_id"], val.index)}
per = pd.DataFrame({"bm25": cp.per_query(run_bm25, j)["ndcg@10"],
                    "dense": cp.per_query(run_dense, j)["ndcg@10"]})
per = per.join(val.set_index("query_id")[["query", "source", "qtype"]]).dropna(subset=["bm25", "dense"])
per["difference"] = per["dense"] - per["bm25"]
print(f"{len(per)} validation queries with an NDCG: bm25 {per['bm25'].mean():.3f}, dense {per['dense'].mean():.3f}")

> **Check.** 121 queries; BM25 about 0.533, dense about 0.458.

## 2. Your turn: the difference by source

`per["difference"]` is dense minus BM25 for each query. Compute its mean for each source, and the source where dense search loses most.

In [ ]:
by_source = ...   # the mean of per["difference"] for each source (groupby), rounded to 3 decimals
print(by_source)
worst_source = ...   # the source with the lowest mean difference
print("dense search loses most on:", worst_source)

In [ ]:
expect_hash('by_source', by_source.to_dict() if by_source is not ... else None, 'e97e638854acde84')

In [ ]:
expect('worst_source', worst_source, 'translated')

> **Check.** Here, dense search loses most on the French queries: BM25 matches French words in the French titles exactly. On the misspelt queries the difference is small (-0.039), because this plain BM25 does not correct spelling either. In the lesson, the project's BM25 corrects spelling first, and dense search loses 0.256 on misspelt test queries. A slice result depends on what you compare against.

The same by kind of query. Run the next cell:

In [ ]:
by_type = per.groupby("qtype")["difference"].agg(["count", "mean"]).round(3)
print(by_type)

> **Look.** Broad queries: -0.198. Short queries give the model little meaning to work with.

## 3. The worst and the best queries

Averages tell you where to look. The queries tell you why. Run the next cell and read both lists.

In [ ]:
print("Dense search loses most:")
print(per.sort_values("difference").head(5)[["query", "source", "bm25", "dense"]].round(3).to_string())
print()
print("Dense search wins most:")
print(per.sort_values("difference", ascending=False).head(5)[["query", "source", "bm25", "dense"]].round(3).to_string())

> **Think.** For each of the five worst queries, is it a name, a misspelling, a code, or a short query? For the five best, why did BM25 miss?

## 4. Your turn: product codes without the rule

The project keeps an exact-match rule for codes in every run. Here there is none. Count how often dense search alone puts the right product first for the 10 code queries of validation.

In [ ]:
codes = val[val["source"] == "code"]
first_right = 0
for qid, row in zip(codes["query_id"], codes.index):
    ranked = ...   # dense search alone: the top 10 of this query
    ...            # add 1 to first_right when the first product has grade 2 (exact) in j[qid]
print(f"dense search alone puts the right product first for {first_right} of {len(codes)} code queries")

In [ ]:
expect('first_right', first_right, 0)

> **Check.** 0 of 10. A code has no meaning that a model can learn. Keep the rule in front of every method.

In the project: `search compare bm25-norm+spell+syn dense --by source` and `--by qtype`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Hybrid search](https://learning.nextia-ai.com/courses/search/m05/hybrid-search/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m05/where-dense-search-fails/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.